# E-commerce Sales Analytics\n\nCapstone project notebook. Start with the raw dataset, inspect data quality, clean it, then explore trends and generate charts. The included demo file is simulated for reproducibility; replace it with the public Sample Superstore dataset for final submission.

In [ ]:
from pathlib import Path\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nraw_path = ROOT / 'data' / 'raw' / 'sales_data.csv'\nif not raw_path.exists(): raw_path = ROOT / 'data' / 'raw' / 'sales_data_demo.csv'\ndf = pd.read_csv(raw_path)\nprint('Shape:', df.shape)\ndf.head()

## 1. Data inspection and data quality

In [ ]:
df.info()\nprint('\\nMissing values:\\n', df.isna().sum())\nprint('Duplicate rows:', df.duplicated().sum())\ndf.describe(include='all').T

## 2. Cleaning

In [ ]:
df.columns = df.columns.str.strip()\nfor col in ['Order Date', 'Ship Date']:\n    if col in df.columns: df[col] = pd.to_datetime(df[col], errors='coerce')\nfor col in ['Sales','Quantity','Discount','Profit']:\n    if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')\ndf = df.drop_duplicates()\ndf = df[df['Sales'].notna() & (df['Sales'] >= 0) & df['Quantity'].notna() & (df['Quantity'] > 0)]\nfor col in ['Segment','Category','Region','Sub-Category']:\n    if col in df.columns and df[col].isna().any(): df[col] = df[col].fillna(df[col].mode().iloc[0])\nif 'City' in df.columns: df['City'] = df['City'].fillna('Unknown')\nif 'Discount' in df.columns: df['Discount'] = df['Discount'].fillna(df['Discount'].median())\ndf['Order Year'] = df['Order Date'].dt.year\ndf['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)\ndf['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], 0)\nprint('Cleaned shape:', df.shape)

## 3. Descriptive statistics and business summaries

In [ ]:
df[['Sales','Quantity','Discount','Profit']].describe().round(2)

In [ ]:
print('Sales by category\\n', df.groupby('Category')['Sales'].sum().sort_values(ascending=False))\nprint('\\nProfit by region\\n', df.groupby('Region')['Profit'].sum().sort_values(ascending=False))\nprint('\\nSales by segment\\n', df.groupby('Segment')['Sales'].sum().sort_values(ascending=False))

## 4. Correlation and outliers

In [ ]:
corr = df[['Sales','Quantity','Discount','Profit']].corr()\ndisplay(corr.round(2))\ncorr.style.background_gradient(cmap='coolwarm', axis=None)

In [ ]:
for col in ['Sales','Profit','Quantity']:\n    q1, q3 = df[col].quantile([0.25, 0.75])\n    iqr = q3 - q1\n    count = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()\n    print(f'{col}: {count} potential IQR outliers')

## 5. Visualizations

In [ ]:
monthly = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()\nmonthly.plot(title='Monthly Sales Trend', figsize=(10,4)); plt.xlabel('Month'); plt.ylabel('Sales'); plt.tight_layout(); plt.show()\ndf.groupby('Category')['Sales'].sum().sort_values().plot(kind='barh', title='Sales by Category', figsize=(8,4)); plt.xlabel('Sales'); plt.tight_layout(); plt.show()\ndf.groupby('Region')['Profit'].sum().sort_values().plot(kind='bar', title='Profit by Region', figsize=(8,4)); plt.ylabel('Profit'); plt.tight_layout(); plt.show()\ndf.groupby('Sub-Category')['Sales'].sum().nlargest(8).sort_values().plot(kind='barh', title='Top Sub-Categories by Sales', figsize=(8,5)); plt.xlabel('Sales'); plt.tight_layout(); plt.show()\ndf.groupby('Segment')['Sales'].sum().plot(kind='pie', autopct='%1.1f%%', title='Sales Mix by Segment', figsize=(6,6)); plt.ylabel(''); plt.show()

## 6. Findings and limitations\n\nUse the summaries and charts above to write five insights based on the actual public dataset. Do not submit the simulated demo data as if it were a real business dataset. The included charts are examples generated from the demo file and should be regenerated after replacing the source CSV.